# 🏛️ CELL 0: Tổng quan Hệ thống UIT Legal Information Retrieval (Chuẩn SoICT / UIT)

## Pipeline: BM25 + Multi-Dense Embeddings → PhoRanker Cross-Encoder Re-ranking
**Kiến trúc dựa trên nghiên cứu chính thức của BTC (arXiv:2507.14619v1 — Team 4Huiter, Top 3 SoICT Hackathon)**

### Datasets & Cache inputs cần thêm vào Notebook trên Kaggle:
| Dữ liệu / Cache | Đường dẫn Kaggle |
|:---|:---|
| Dữ liệu luật & câu hỏi | `/kaggle/input/datasets/thurdayafternoon/uit-legal-ir-data/uit-legal-ir-data` |
| Mô hình Bi-Encoder đã fine-tune | `/kaggle/input/datasets/thurdayafternoon/uit-legal-finetuned-model/fine_tuned_vietnamese_bi_encoder` |
| Output từ Notebook trước (PKL Cache) | `/kaggle/input/notebooks/thurdayafternoon/legal-ir` |
| Cache PKL (Dataset riêng nếu có) | `/kaggle/input/datasets/thurdayafternoon/pkl-cache` |



In [ ]:
# ==============================================================================
# 📌 CELL 1: Cài đặt Môi trường & Các Thư viện Phụ thuộc (Dependencies)
# ==============================================================================
import os
import subprocess
import re

# ⚠️ Khóa về Single GPU ngay từ đầu để tránh lỗi PyTorch DataParallel ('tokenizer' attribute error trên T4x2)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# Kiểm tra GPU
try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"], text=True).strip()
    print(f"🎮 GPU: {gpu_info}")
    cap = float(re.search(r'(\d+\.\d+)', gpu_info.split(',')[-1]).group(1))
    if cap < 7.0:
        print("⚠️ GPU compute capability < 7.0 (P100). Cài PyTorch CUDA 11.8...")
        !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
    else:
        print("✅ GPU tương thích với PyTorch CUDA mặc định.")
except Exception as e:
    print(f"ℹ️ Thông tin GPU: {e}")

# Cài đặt các thư viện cần thiết
!pip install -q sentence-transformers rank-bm25 pyvi scikit-learn matplotlib tqdm



In [ ]:
# ==============================================================================
# 📌 CELL 2: Clone GitHub Repository & Thiết lập Môi trường Làm việc Kaggle
# ==============================================================================
import os
import shutil

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)

# === 1. CLONE CODE TỪ GITHUB ===
REPO_URL = "https://github.com/manh123-chatgpt/implement-pp1.git"
CLONE_DIR = os.path.join(WORK_DIR, "code")

if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)

print("📦 Đang clone code từ GitHub...")
os.system(f"git clone {REPO_URL} {CLONE_DIR}")

# Copy toàn bộ file .py từ repo vào working dir
if os.path.exists(CLONE_DIR):
    for f in os.listdir(CLONE_DIR):
        if f.endswith(".py"):
            src = os.path.join(CLONE_DIR, f)
            dst = os.path.join(WORK_DIR, f)
            shutil.copy2(src, dst)
            print(f"  ✅ {f}")

# === 2. CẤU HÌNH ĐƯỜNG DẪN KAGGLE DATASETS & NOTEBOOK OUTPUTS ===
KAGGLE_DATA_DIR = "/kaggle/input/datasets/thurdayafternoon/uit-legal-ir-data/uit-legal-ir-data"
KAGGLE_MODEL_DIR = "/kaggle/input/datasets/thurdayafternoon/uit-legal-finetuned-model/fine_tuned_vietnamese_bi_encoder"

# Danh sách các thư mục nguồn để tìm kiếm file cache PKL và outputs
KAGGLE_CACHE_DIRS = [
    "/kaggle/input/notebooks/thurdayafternoon/legal-ir",       # Output từ notebook trước
    "/kaggle/input/datasets/thurdayafternoon/pkl-cache",       # Dataset upload riêng
]

# === 3. COPY / SYMLINK DỮ LIỆU VÀO WORKING DIR ===

# 3a. Copy legal_corpus_resolved.json
RESOLVED_SRC = os.path.join(KAGGLE_DATA_DIR, "legal_corpus_resolved.json")
RESOLVED_DST = os.path.join(WORK_DIR, "legal_corpus_resolved.json")
if os.path.exists(RESOLVED_SRC) and not os.path.exists(RESOLVED_DST):
    print("📄 Đang copy legal_corpus_resolved.json...")
    shutil.copy2(RESOLVED_SRC, RESOLVED_DST)
    print(f"  ✅ {os.path.getsize(RESOLVED_DST) / 1024**2:.0f} MB")

# 3b. Symlink thư mục fine_tuned_vietnamese_bi_encoder
MODEL_LINK = os.path.join(WORK_DIR, "fine_tuned_vietnamese_bi_encoder")
if not os.path.exists(MODEL_LINK) and os.path.exists(KAGGLE_MODEL_DIR):
    os.symlink(KAGGLE_MODEL_DIR, MODEL_LINK)
    print(f"🔗 Symlink mô hình Bi-Encoder: {MODEL_LINK} → {KAGGLE_MODEL_DIR}")

# 3c. Copy/Symlink các file cache PKL (tự động dò tìm trong các nguồn)
PKL_FILES = [
    "bm25_tokenized_cache_pyvi_v2.pkl",
    "corpus_embeddings_bgem3_resolved.pkl",
    "corpus_embeddings_e5_resolved.pkl",
    "corpus_embeddings_finetuned_resolved.pkl",
    "semi_hard_negatives.pkl",
]

for pkl in PKL_FILES:
    dst = os.path.join(WORK_DIR, pkl)
    if os.path.exists(dst):
        continue
    for cache_dir in KAGGLE_CACHE_DIRS:
        src = os.path.join(cache_dir, pkl)
        if os.path.exists(src) and os.path.getsize(src) > 1024:
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(dst) / 1024**2
            print(f"  ✅ {pkl} ({size_mb:.0f} MB) ← {cache_dir}")
            break
    else:
        print(f"  ⚠️ {pkl} — Chưa có (sẽ tự động tính nếu cần)")

# 3d. Copy mine_semi_hard_negatives.py từ notebook output nếu có
for cache_dir in KAGGLE_CACHE_DIRS:
    src_script = os.path.join(cache_dir, "mine_semi_hard_negatives.py")
    if os.path.exists(src_script) and not os.path.exists(os.path.join(WORK_DIR, "mine_semi_hard_negatives.py")):
        shutil.copy2(src_script, os.path.join(WORK_DIR, "mine_semi_hard_negatives.py"))
        print(f"  ✅ mine_semi_hard_negatives.py ← {cache_dir}")
        break

# 3e. Copy fine_tuned_vietnamese_cross_encoder nếu có (Dùng copytree thay vì symlink để tránh lỗi Read-only)
for cache_dir in KAGGLE_CACHE_DIRS:
    ce_src = os.path.join(cache_dir, "fine_tuned_vietnamese_cross_encoder")
    ce_dst = os.path.join(WORK_DIR, "fine_tuned_vietnamese_cross_encoder")
    if os.path.isdir(ce_src) and not os.path.exists(ce_dst):
        try:
            shutil.copytree(ce_src, ce_dst)
            print(f"  ✅ Đã copy mô hình Cross-Encoder vào working: {ce_dst}")
        except Exception as e:
            print(f"  ⚠️ Không thể copy Cross-Encoder: {e}")
        break

# === 4. TỰ ĐỘNG VÁ LỖI AN TOÀN (HOTFIX) TRONG CODE ===
# Đảm bảo 'import torch' luôn có ở đầu hybrid_retriever.py (tránh lỗi name 'torch' is not defined)
hybrid_file = os.path.join(WORK_DIR, "hybrid_retriever.py")
if os.path.exists(hybrid_file):
    with open(hybrid_file, "r", encoding="utf-8") as f:
        code = f.read()
    if "import torch" not in code[:300]:
        with open(hybrid_file, "w", encoding="utf-8") as f:
            f.write("import torch\n" + code)
        print("  🛡️ Hotfix: Đã bổ sung 'import torch' vào đầu hybrid_retriever.py")

print("\n" + "=" * 60)
print("🎯 SETUP HOÀN TẤT! Sẵn sàng chạy pipeline.")
print("=" * 60)



In [ ]:
# ==============================================================================
# 📌 CELL 3: Kiểm tra Phần cứng GPU & Bảng Điều Khiển Trung Tâm (Central Controller)
# ==============================================================================
import torch
import os

# Kiểm tra GPU
print("🖥️ THÔNG TIN HỆ THỐNG:")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB VRAM)")

# Kiểm tra file đã sẵn sàng
print("\n📂 KIỂM TRA FILE TRONG WORKSPACE:")
check_files = {
    "data_loader.py": "Code - Data Loader",
    "bm25_retriever.py": "Code - BM25 Searcher",
    "dense_retriever.py": "Code - Dense Retriever",
    "hybrid_retriever.py": "Code - Hybrid 2-Stage Searcher",
    "evaluator.py": "Code - Evaluator",
    "mine_semi_hard_negatives.py": "Code - Semi-Hard Negative Mining",
    "train_cross_encoder.py": "Code - Cross-Encoder Trainer",
    "legal_corpus_resolved.json": "Data - Graph Resolved Corpus",
    "fine_tuned_vietnamese_bi_encoder": "Model - Fine-tuned Bi-Encoder",
    "fine_tuned_vietnamese_cross_encoder": "Model - Fine-tuned Cross-Encoder",
    "bm25_tokenized_cache_pyvi_v2.pkl": "Cache - BM25 Tokenized",
    "corpus_embeddings_finetuned_resolved.pkl": "Cache - Bi-Encoder Embeddings",
    "corpus_embeddings_bgem3_resolved.pkl": "Cache - BGE-M3 Embeddings",
    "corpus_embeddings_e5_resolved.pkl": "Cache - E5-Large Embeddings",
    "semi_hard_negatives.pkl": "Cache - Semi-Hard Negatives",
}
for fname, desc in check_files.items():
    exists = "✅" if os.path.exists(fname) else "❌"
    size_str = f" ({os.path.getsize(fname)/1024**2:.1f} MB)" if os.path.exists(fname) and os.path.isfile(fname) else ""
    print(f"  {exists} {desc}: {fname}{size_str}")

# ==============================================================================
# 🎛️ BẢNG ĐIỀU KHIỂN TRUNG TÂM (CENTRAL PIPELINE CONTROLLER)
# ==============================================================================

# 1. ⚡ CHẾ ĐỘ BUILD PKL CACHE (Dùng khi "Save Version" chỉ để xuất file .pkl):
#    - True : Chỉ build & lưu toàn bộ file .pkl (BM25 + Dense Embeddings + Clean Corpus).
#             Bỏ qua toàn bộ đánh giá, train model, grid search, sinh submission.
#             Chạy xong cực nhanh (~3 phút), tạo Output .pkl artifacts hoàn hảo để tải về hoặc dùng lại!
#    - False: Chạy pipeline bình thường (Interactive hoặc Full Commit).
BUILD_PKL_CACHE_ONLY = False

# 2. ⚡ TÙY CHỌN CÁC FILE CACHE .PKL CẦN TẠO KHI BUILD_PKL_CACHE_ONLY = True:
BUILD_BM25_PKL = True            # bm25_tokenized_cache_pyvi_v2.pkl (~120 MB)
BUILD_BIENCODER_PKL = True       # corpus_embeddings_finetuned_resolved.pkl (~85 MB)
BUILD_BGEM3_PKL = False          # Bật True nếu muốn build trước cache BGE-M3
BUILD_E5_PKL = False             # Bật True nếu muốn build trước cache E5-Large
BUILD_QWEN4B_PKL = False         # Bật True nếu muốn build trước cache Qwen3-4B
BUILD_SEMI_HARD_NEG_PKL = False  # Bật True nếu muốn khai thác trước semi_hard_negatives.pkl

# 3. ⚡ BẬT/TẮT SINH SUBMISSION BASELINE (STAGE 1):
#    - False (Mặc định): Chạy thẳng, KHÔNG sinh submission.zip baseline để dồn tài nguyên cho Grid Search & Reranker!
#    - True : Dự đoán 1000 câu Public Test tạo submission.zip (Stage 1).
GENERATE_BASELINE = False

# 4. ⚡ BẬT/TẮT ĐÁNH GIÁ 500 CÂU VALIDATION Ở CELL 6:
#    - False (Mặc định): Bỏ qua đánh giá Cell 6 để chạy thẳng siêu tốc!
#    - True : Đánh giá 500 câu validation ở Cell 6 để xem điểm Recall/Precision ban đầu.
EVALUATE_CELL_6 = False

# 5. ⚡ BẬT/TẮT HUẤN LUYỆN CROSS-ENCODER (CELL 10 & 11):
#    - False (Mặc định): Bỏ qua (nếu đã có checkpoint hoặc dùng Zero-shot Qwen3-Reranker-4B).
#    - True : Khai thác Semi-Hard Negatives và train PhoRanker.
TRAIN_CROSS_ENCODER = False

# 6. 👑 CHỌN MÔ HÌNH RERANKER (STAGE 2):
#    - "Qwen/Qwen3-Reranker-4B": Mô hình mạnh nhất (4B params, 2048 context, FP16) [KHUYÊN DÙNG]
#    - "fine_tuned_vietnamese_cross_encoder": PhoRanker đã fine-tuned
#    - "itdainb/PhoRanker": PhoRanker gốc
CHOSEN_RERANKER_MODEL = "Qwen/Qwen3-Reranker-4B"

print("\n" + "=" * 65)
print("🎛️ TRẠNG THÁI CẤU HÌNH HIỆN TẠI:")
print(f"  👉 BUILD_PKL_CACHE_ONLY : {BUILD_PKL_CACHE_ONLY}")
print(f"  👉 GENERATE_BASELINE    : {GENERATE_BASELINE}")
print(f"  👉 EVALUATE_CELL_6      : {EVALUATE_CELL_6}")
print(f"  👉 TRAIN_CROSS_ENCODER  : {TRAIN_CROSS_ENCODER}")
print(f"  👉 CHOSEN_RERANKER_MODEL: {CHOSEN_RERANKER_MODEL}")
print("=" * 65)




In [ ]:
# ==============================================================================
# 📌 CELL 4: Nạp Dữ liệu Pháp luật (Clean Corpus, Train, Validation 500, Test)
# ==============================================================================
from data_loader import load_corpus, load_train_data, load_test_data

corpus = load_corpus()
train_data = load_train_data()
test_data = load_test_data()

# Chuẩn bị sẵn tập Validation 500 câu (để Cell 6 hoặc Cell 15 dùng ngay)
val_items = list(train_data.items())[:500]
val_questions = {k: v["question"] for k, v in val_items}
val_truth = {k: v["answer"] for k, v in val_items}

print(f"\n📊 Corpus     : {len(corpus)} văn bản pháp luật")
print(f"📊 Train      : {len(train_data)} câu hỏi")
print(f"📊 Validation : {len(val_questions)} câu hỏi")
print(f"📊 Public Test: {len(test_data)} câu hỏi")



## 🔍 CELL 5: Giới thiệu Stage 1 + Stage 2 Hybrid 2-Stage Retrieval & Re-ranking
- **Stage 1**: BM25 (Lexical) + Dense Bi-Encoders → Lọc **Top 90** ứng viên
- **Stage 2**: PhoRanker Cross-Encoder re-rank 90 ứng viên → Chọn **Top 5** chính xác nhất

> ⚡ **Tự động tối ưu chế độ:**
>- Nếu **đã có đủ 3 cache embeddings** (`finetuned`, `bgem3`, `e5`), notebook tự động chạy **Full Ensemble Mode (`light_mode=False`)** để kết hợp toàn bộ sức mạnh của 4 mô hình, cho Recall cao nhất mà chỉ mất vài giây để nạp!
>- Nếu **chưa đủ cache**, notebook tự động chạy **`light_mode=True`** (BM25 + Bi-Encoder + PhoRanker) để đảm bảo tốc độ nhanh (< 15 phút).



In [ ]:
# ==============================================================================
# 📌 CELL 6: Khởi tạo Bộ tìm kiếm HybridSearcher (Build / Load Cache .PKL)
# ==============================================================================
import os
from hybrid_retriever import HybridSearcher
from evaluator import compute_metrics
from tqdm import tqdm

# Kiểm tra các file cache embeddings có tồn tại và đầy đủ dung lượng (> 5 MB) không
def is_valid_cache(fname, min_mb=5):
    return os.path.exists(fname) and (os.path.getsize(fname) / 1024**2 >= min_mb)

has_all_dense_caches = (
    is_valid_cache("corpus_embeddings_bgem3_resolved.pkl", min_mb=10) and
    is_valid_cache("corpus_embeddings_e5_resolved.pkl", min_mb=10) and
    is_valid_cache("corpus_embeddings_finetuned_resolved.pkl", min_mb=10)
)

# Tự động chọn chế độ an toàn & tối ưu nhất
use_light = not has_all_dense_caches
if has_all_dense_caches:
    print("🔥 Đã phát hiện đầy đủ 3 cache embeddings hợp lệ (BGE-M3 + E5-Large + Bi-Encoder)!")
    print("🚀 Kích hoạt FULL ENSEMBLE MODE (light_mode=False) để tối đa hóa điểm số!")
else:
    print("⚡ Chưa đủ hoặc thiếu cache embeddings. Tự động chạy LIGHT MODE (BM25 + Bi-Encoder).")
    print("   (Chế độ này an toàn tuyệt đối, Recall@5 vẫn đạt ~90% và thời gian chạy cực nhanh!)")

# Khởi tạo hệ thống 2-Stage chuẩn BTC UIT (Tự động tạo BM25 PKL & Bi-Encoder PKL nếu chưa có)
has_ft_cross = os.path.exists("fine_tuned_vietnamese_cross_encoder")
hybrid = HybridSearcher(corpus, use_reranker=has_ft_cross, light_mode=use_light)

# Tùy chọn build thêm các cache dense models khác nếu được bật:
if BUILD_BGEM3_PKL and not is_valid_cache("corpus_embeddings_bgem3_resolved.pkl"):
    from dense_retriever import BGEEmbeddingSearcher
    print("📦 Đang khởi tạo và lưu cache BGE-M3...")
    _ = BGEEmbeddingSearcher(corpus)

if BUILD_E5_PKL and not is_valid_cache("corpus_embeddings_e5_resolved.pkl"):
    from dense_retriever import E5EmbeddingSearcher
    print("📦 Đang khởi tạo và lưu cache E5-Large...")
    _ = E5EmbeddingSearcher(corpus)

if BUILD_QWEN4B_PKL and not is_valid_cache("corpus_embeddings_qwen3_4b_resolved.pkl"):
    from dense_retriever import Qwen4BEmbeddingSearcher
    print("📦 Đang khởi tạo và lưu cache Qwen3-4B...")
    _ = Qwen4BEmbeddingSearcher(corpus)

# Hiển thị danh sách các file .pkl đã tạo trong thư mục hiện tại
print("\n" + "=" * 65)
print("📂 DANH SÁCH FILE ARTIFACTS / CACHE HIỆN CÓ:")
for f in sorted(os.listdir(".")):
    if f.endswith(".pkl") or f.endswith(".json"):
        size_mb = os.path.getsize(f) / (1024**2)
        print(f"  ✅ {f:45s} : {size_mb:6.1f} MB")
print("=" * 65)

# Kiểm tra chế độ BUILD_PKL_CACHE_ONLY
if BUILD_PKL_CACHE_ONLY:
    print("\n" + "=" * 75)
    print("🎉 ĐÃ HOÀN TẤT BUILD TẤT CẢ FILE .PKL CACHE!")
    print("⏭️ Chế độ BUILD_PKL_CACHE_ONLY = True: DỪNG TẠI ĐÂY để hoàn tất Save Version nhanh chóng.")
    print("👉 Các bước sau (đánh giá, train, grid search, nộp bài) bạn có thể chạy trực tiếp!")
    print("=" * 75)
else:
    # Đánh giá 500 câu Validation nếu bật EVALUATE_CELL_6
    if not EVALUATE_CELL_6:
        print("\n⏭️ ĐÃ BỎ QUA đánh giá 500 câu Validation ở Cell 6 (EVALUATE_CELL_6 = False).")
        print("🚀 Đang chạy thẳng xuống các bước tiếp theo...")
    else:
        print("\n🔍 Đang đánh giá trên 500 câu hỏi validation...")
        val_preds = {}
        for qid, question in tqdm(val_questions.items(), desc="2-Stage Validation"):
            val_preds[qid] = hybrid.search(question, top_k=5)

        results = compute_metrics(val_preds, val_truth, k=5)
        print("\n" + "=" * 50)
        print(f"📊 KẾT QUẢ 2-STAGE HYBRID (Light Mode: {use_light}):")
        print(f"👉 Recall@5   : {results['Recall'] * 100:.2f}%")
        print(f"👉 Precision@5: {results['Precision'] * 100:.2f}%")
        print("=" * 50)




## 📦 CELL 7: Hướng dẫn Tạo file Submission cho Public Test (1000 câu hỏi)



In [ ]:
# ==============================================================================
# 📌 CELL 8: Tạo Submission Baseline Stage 1 (Tùy chọn Bật/Tắt qua GENERATE_BASELINE)
# ==============================================================================
# Điều khiển bởi biến GENERATE_BASELINE trong Bảng Điều Khiển Trung Tâm (Cell 3)

if BUILD_PKL_CACHE_ONLY or not GENERATE_BASELINE:
    print("⏭️ ĐÃ BỎ QUA sinh 'submission.zip' baseline (GENERATE_BASELINE = False).")
    print("🚀 Chạy thẳng xuống các bước tiếp theo (Grid Search & Submission Reranker)!")
else:
    import json
    import zipfile
    from tqdm import tqdm

    # Dự đoán trên toàn bộ các câu hỏi Public Test
    print(f"📦 Đang sinh kết quả cho {len(test_data)} câu hỏi Public Test...")
    test_preds = {}
    for qid, item in tqdm(test_data.items(), desc="Predicting Public Test"):
        question = item["question"]
        test_preds[qid] = hybrid.search(question, top_k=5)

    # Tạo file submission chuẩn format BTC
    formatted_preds = {}
    for qid, docs in test_preds.items():
        formatted_preds[str(qid)] = {
            "answer": [str(d) for d in docs[:5]]
        }

    with open("submission.json", "w", encoding="utf-8") as f:
        json.dump(formatted_preds, f, ensure_ascii=False, indent=4)

    with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write("submission.json", arcname="submission.json")

    print(f"\n🎉 Đã tạo thành công file nộp bài: 'submission.zip' ({os.path.getsize('submission.zip') / 1024:.1f} KB)")
    print(f"📊 Tổng số câu hỏi đã dự đoán: {len(formatted_preds)}")

    # Xem thử 3 câu đầu tiên
    for i, (qid, pred) in enumerate(list(formatted_preds.items())[:3]):
        q_text = test_data[qid]["question"][:80]
        print(f"\n  [{qid}] {q_text}...")
        print(f"  → Dự đoán: {pred['answer']}")



## 🔧 CELL 9: Giới thiệu Huấn luyện nâng cao (Semi-Hard Negatives + Cross-Encoder)
> ⚠️ Chỉ chạy nếu muốn huấn luyện lại Cross-Encoder từ đầu (mỗi bước ~15-30 phút GPU).

**Bước 1**: Khai thác mẫu âm bán khó (Semi-Hard Negatives) từ Top 90 Bi-Encoder (Nếu đã có `semi_hard_negatives.pkl` thì tự động bỏ qua).  
**Bước 2**: Huấn luyện PhoRanker Cross-Encoder với BCEWithLogitsLoss (2 epochs) ➔ **Tự động sinh file `submission_finetuned.zip`**!



In [ ]:
# ==============================================================================
# 📌 CELL 10: Khai thác Semi-Hard Negatives (MAX ~87-89 mẫu/câu)
# ==============================================================================
import os

if BUILD_PKL_CACHE_ONLY:
    if BUILD_SEMI_HARD_NEG_PKL:
        print("📦 Đang khai thác Semi-Hard Negatives cho cache PKL...")
        from mine_semi_hard_negatives import mine_semi_hard_negatives
        mine_semi_hard_negatives()
    else:
        print("⏭️ Bỏ qua khai thác Negatives (BUILD_PKL_CACHE_ONLY = True & BUILD_SEMI_HARD_NEG_PKL = False).")
elif not TRAIN_CROSS_ENCODER:
    print("⏭️ Bỏ qua khai thác Negatives (TRAIN_CROSS_ENCODER = False).")
else:
    # ⚠️ QUAN TRỌNG: Phải khai thác LẠI nếu đã tăng num_negatives lên MAX!
    neg_file = "semi_hard_negatives.pkl"
    if os.path.exists(neg_file):
        import pickle
        with open(neg_file, "rb") as f:
            old_negs = pickle.load(f)
        avg_negs = sum(len(v) for v in old_negs.values()) / max(1, len(old_negs))
        if avg_negs < 70:
            print(f"⚠️ File '{neg_file}' cũ chỉ có trung bình {avg_negs:.0f} mẫu/câu. Cần khai thác lại MAX!")
            os.remove(neg_file)
            from mine_semi_hard_negatives import mine_semi_hard_negatives
            mine_semi_hard_negatives()
        else:
            print(f"✅ '{neg_file}' đã có {avg_negs:.0f} mẫu/câu (MAX). Đủ tốt, bỏ qua khai thác lại!")
    else:
        from mine_semi_hard_negatives import mine_semi_hard_negatives
        mine_semi_hard_negatives()



In [ ]:
# ==============================================================================
# 📌 CELL 11: Huấn luyện Cross-Encoder (PhoRanker) & Nạp mô hình Re-ranker
# ==============================================================================
import os

if BUILD_PKL_CACHE_ONLY or not TRAIN_CROSS_ENCODER:
    print("⏭️ Đã bỏ qua huấn luyện Cross-Encoder (BUILD_PKL_CACHE_ONLY hoặc TRAIN_CROSS_ENCODER = False).")
else:
    from train_cross_encoder import train_cross_encoder
    import json
    import zipfile
    from tqdm import tqdm
    from hybrid_retriever import HybridSearcher

    FT_MODEL_DIR = "fine_tuned_vietnamese_cross_encoder"
    has_valid_ce = os.path.exists(FT_MODEL_DIR) and os.path.exists(os.path.join(FT_MODEL_DIR, "config.json"))
    FORCE_RETRAIN_CE = False

    if has_valid_ce and not FORCE_RETRAIN_CE:
        print("=" * 65)
        print(f"⚡ ĐÃ PHÁT HIỆN MÔ HÌNH CROSS-ENCODER CÓ SẴN TẠI '{FT_MODEL_DIR}'!")
        print("👉 Bỏ qua bước huấn luyện để TIẾT KIỆM 8 TIẾNG GPU, dùng luôn mô hình này!")
        print("=" * 65)
    else:
        if os.path.islink(FT_MODEL_DIR):
            os.unlink(FT_MODEL_DIR)
        train_cross_encoder()

    if os.path.exists(FT_MODEL_DIR):
        print("\n" + "=" * 65)
        print("🚀 KHỞI ĐỘNG RETRIEVER VỚI 3 FIX CỐT LÕI:")
        print("   1. Pyvi tokenization đồng bộ Train ↔ Inference")
        print("   2. Fusion weights: 0.75 * Stage1 + 0.25 * Reranker")
        print("   3. Re-rank Top 30 ứng viên tinh hoa")
        print("=" * 65)
        hybrid_ft = HybridSearcher(corpus, use_reranker=True, light_mode=use_light, reranker_model_path=FT_MODEL_DIR)



## 🔬 CELL 12: (Tùy chọn) Grid Search: Cross-Encoder Training Hyperparameters

> ⚠️ **TỐN THỜI GIAN** (~30-48 phút Preset B, ~4-6 giờ Preset A).
> 
> Nếu **bỏ qua** cell này, hệ thống dùng model từ Cell 13 (mặc định: ep=2, lr=2e-5, bs=32).
> 
> Nếu **chạy** cell này, model tốt nhất được lưu vào `fine_tuned_vietnamese_cross_encoder/`
> và Cell 17 (Score Fusion GS) sẽ tự động dùng nó.



In [ ]:
# ==============================================================================
# 📌 CELL 13: Grid Search Cross-Encoder Training (Tùy chọn Bỏ qua nếu dùng Model 4B)
# ==============================================================================
if BUILD_PKL_CACHE_ONLY or not TRAIN_CROSS_ENCODER:
    print("⏭️ Đã bỏ qua CE Training Grid Search (BUILD_PKL_CACHE_ONLY hoặc TRAIN_CROSS_ENCODER = False).")
else:
    # === GRID SEARCH: Cross-Encoder Training Hyperparameters ===
    # ⚠️ TỐN THỜI GIAN! Mỗi config train ~5-8 phút trên T4 GPU.
    
    # 🎛️ ĐẶT SKIP_CE_GS = True NẾU MUỐN BỎ QUA (dùng model mặc định từ Cell 13)
    SKIP_CE_GS = False  # ← Đổi thành True để skip
    
    if SKIP_CE_GS:
        print("⏭️ Bỏ qua CE Grid Search. Dùng model mặc định từ Cell 13.")
    else:
    
        import os, shutil, gc, torch
        from train_cross_encoder import train_cross_encoder
        from hybrid_retriever import HybridSearcher
        from evaluator import compute_metrics
        from tqdm import tqdm
        
        # === GRID SEARCH PARAMETERS ===
        # Chọn MỘT trong 2 preset bên dưới:
        
        # Preset A: ĐẦY ĐỦ (~48 configs, ~4-6 giờ)
        # CE_EPOCHS_GRID = [1, 2, 3, 4]
        # CE_LR_GRID = [1e-5, 2e-5, 3e-5, 5e-5]
        # CE_BS_GRID = [16, 32, 64]
        
        # Preset B: NHANH (~6 configs, ~30-48 phút) ← KHUYÊN DÙNG
        CE_EPOCHS_GRID = [2, 3]
        CE_LR_GRID = [1e-5, 2e-5, 3e-5]
        CE_BS_GRID = [32]
        
        FT_MODEL_DIR = "fine_tuned_vietnamese_cross_encoder"
        TEMP_MODEL_DIR = "temp_ce_gridsearch"
        
        print("=" * 80)
        print("🔬 GRID SEARCH: CROSS-ENCODER TRAINING HYPERPARAMETERS")
        print(f"   Epochs: {CE_EPOCHS_GRID}")
        print(f"   LR:     {CE_LR_GRID}")
        print(f"   BS:     {CE_BS_GRID}")
        total_configs = len(CE_EPOCHS_GRID) * len(CE_LR_GRID) * len(CE_BS_GRID)
        print(f"   Tổng configs: {total_configs}")
        print("=" * 80)
        
        ce_results = []
        best_recall = 0.0
        best_ce_config = {}
        config_idx = 0
        
        for epochs in CE_EPOCHS_GRID:
            for lr in CE_LR_GRID:
                for bs in CE_BS_GRID:
                    config_idx += 1
                    print(f"\n{'='*60}")
                    print(f"🔧 Config {config_idx}/{total_configs}: epochs={epochs}, lr={lr}, bs={bs}")
                    print(f"{'='*60}")
        
                    # Xóa model tạm cũ (nếu có)
                    if os.path.exists(TEMP_MODEL_DIR):
                        shutil.rmtree(TEMP_MODEL_DIR)
        
                    # Train với config hiện tại
                    try:
                        val_loss = train_cross_encoder(
                            epochs=epochs, lr=lr, batch_size=bs,
                            output_dir=TEMP_MODEL_DIR
                        )
                    except Exception as e:
                        print(f"❌ Lỗi training: {e}")
                        ce_results.append({"epochs": epochs, "lr": lr, "bs": bs, "recall": 0, "precision": 0, "val_loss": 999})
                        continue
        
                    # Evaluate trên validation
                    try:
                        hybrid_temp = HybridSearcher(corpus, use_reranker=True, light_mode=use_light, reranker_model_path=TEMP_MODEL_DIR)
                        preds = {}
                        for qid, question in val_questions.items():
                            preds[qid] = hybrid_temp.search(question, top_k=5, rerank_top_k=30, st1_weight=0.75)
                        metrics = compute_metrics(preds, val_truth, k=5)
                        recall = metrics["Recall"]
                        precision = metrics["Precision"]
        
                        # Giải phóng bộ nhớ GPU
                        del hybrid_temp
                        gc.collect()
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                    except Exception as e:
                        print(f"❌ Lỗi evaluation: {e}")
                        recall, precision = 0, 0
        
                    ce_results.append({
                        "epochs": epochs, "lr": lr, "bs": bs,
                        "recall": recall, "precision": precision,
                        "val_loss": val_loss if val_loss else 999
                    })
        
                    marker = ""
                    if recall > best_recall:
                        best_recall = recall
                        best_ce_config = {"epochs": epochs, "lr": lr, "bs": bs, "recall": recall, "precision": precision}
                        # Lưu model tốt nhất vào FT_MODEL_DIR
                        if os.path.exists(FT_MODEL_DIR):
                            shutil.rmtree(FT_MODEL_DIR)
                        shutil.copytree(TEMP_MODEL_DIR, FT_MODEL_DIR)
                        marker = " ⭐ NEW BEST! (saved)"
        
                    print(f"📊 Recall={recall*100:.2f}% | Precision={precision*100:.2f}% | Val Loss={val_loss:.4f}{marker}")
        
        # Dọn dẹp
        if os.path.exists(TEMP_MODEL_DIR):
            shutil.rmtree(TEMP_MODEL_DIR)
        
        # === Bảng tổng kết ===
        print("\n" + "=" * 80)
        print("📊 BẢNG TỔNG KẾT CROSS-ENCODER GRID SEARCH")
        print("=" * 80)
        sorted_ce = sorted(ce_results, key=lambda x: x["recall"], reverse=True)
        print(f"{'Config':>30} | {'Recall':>10} | {'Precision':>10} | {'Val Loss':>10}")
        print("-" * 70)
        for r in sorted_ce:
            config_str = f"ep={r['epochs']} lr={r['lr']:.0e} bs={r['bs']}"
            print(f"{config_str:>30} | {r['recall']*100:>8.2f}% | {r['precision']*100:>8.2f}% | {r['val_loss']:>10.4f}")
        
        print(f"\n⭐ CẤU HÌNH TỐI ƯU: epochs={best_ce_config['epochs']}, lr={best_ce_config['lr']}, bs={best_ce_config['bs']}")
        print(f"   → Recall@5: {best_ce_config['recall']*100:.2f}% (đã lưu vào '{FT_MODEL_DIR}')")
        print(f"   → Tiếp tục chạy Cell 15 (Score Fusion Grid Search) với model tốt nhất này!")



## 🔬 CELL 14: Giới thiệu Grid Search Score Fusion & Re-rank Hyperparameters

> **Chạy SAU Cell 13/15** (cần Cross-Encoder đã fine-tune).
> 
> Cell này dùng model CE tốt nhất (từ Cell 13 mặc định hoặc Cell 15 Grid Search)
> để tìm tổ hợp tối ưu `rerank_top_k × st1_weight`.



In [ ]:
# ==============================================================================
# 📌 CELL 15: Grid Search Siêu Tốc Trong Bộ Nhớ (Score Fusion & Re-rank 70 Tổ Hợp)
# ==============================================================================
if BUILD_PKL_CACHE_ONLY:
    print("⏭️ Đã bỏ qua Grid Search vì BUILD_PKL_CACHE_ONLY = True.")
else:
    import os
    import time
    from tqdm import tqdm
    from evaluator import compute_metrics
    from hybrid_retriever import HybridSearcher
    
    FT_MODEL_DIR = "fine_tuned_vietnamese_cross_encoder"
    has_model = os.path.exists(FT_MODEL_DIR) or os.path.exists("/kaggle/input/notebooks/thurdayafternoon/legal-ir/fine_tuned_vietnamese_cross_encoder")
    
    print("=" * 80)
    print("🔬 GRID SEARCH SIÊU TỐC TRONG BỘ NHỚ (IN-MEMORY VECTORIZED FUSION)")
    print("=" * 80)
    
    # ⚡ CHẾ ĐỘ THỬ NGHIỆM SIÊU TỐC (FAST DEV MODE):
    #   - True : Chỉ chạy trên 50 câu hỏi Validation (~15 giây) để kiểm tra cực nhanh trước khi commit!
    #   - False: Chạy trên toàn bộ 500 câu hỏi Validation khi chuẩn bị Save Version chính thức.
    if 'val_questions' not in globals():
        val_items = list(train_data.items())[:500]
        val_questions = {k: v["question"] for k, v in val_items}
        val_truth = {k: v["answer"] for k, v in val_items}

    FAST_DEV_MODE = True
    NUM_VAL_SAMPLES = 50 if FAST_DEV_MODE else len(val_questions)
    
    val_items_eval = dict(list(val_questions.items())[:NUM_VAL_SAMPLES])
    val_truth_eval = {qid: val_truth[qid] for qid in val_items_eval}
    mode_str = "⚡ FAST DEV (50 câu - 15 giây)" if FAST_DEV_MODE else "🔥 FULL VALIDATION (500 câu)"
    print(f"🎯 Đang sử dụng chế độ: {mode_str}")
    
    # 1. Khởi tạo Retriever với mô hình tốt nhất (Hỗ trợ Tier S+ Qwen3-4B / Jina-v3)
    # 👑 TÙY CHỌN MÔ HÌNH RERANKER:
    #   - Đặt RERANK_MODEL = 'Qwen/Qwen3-Reranker-4B' để dùng Quái Vật 4B mạnh nhất giải đấu!
    #   - Đặt RERANK_MODEL = FT_MODEL_DIR để dùng PhoRanker fine-tuned.
    RERANK_MODEL = CHOSEN_RERANKER_MODEL if 'CHOSEN_RERANKER_MODEL' in globals() else 'Qwen/Qwen3-Reranker-4B'
    
    hybrid_gs = HybridSearcher(
        corpus,
        use_reranker=True,
        light_mode=use_light,
        reranker_model_path=RERANK_MODEL,
        use_jina=False,     # Đổi True nếu muốn ensemble Jina-v3 (8k context)
        use_qwen=False,     # Đổi True nếu muốn ensemble Qwen-Legal (4k context)
        use_qwen4b=False    # Đổi True nếu muốn dùng Qwen3-Embedding-4B
    )
    
    # 2. TIỀN TÍNH TOÁN (PRE-COMPUTE) ĐIỂM SỐ ĐÚNG 1 LẦN DUY NHẤT TRÊN VALIDATION
    print(f"\n⚡ [Bước 1/2] Đang tính toán điểm số Stage 1 & Reranker cho {len(val_items_eval)} câu Validation...")
    t0 = time.time()
    cached_val_items = {}
    for qid, question in tqdm(val_items_eval.items(), desc="Pre-computing validation scores"):
        cached_val_items[qid] = hybrid_gs.get_query_candidates_and_rerank_scores(
            question, candidate_k=90, max_rerank_k=50
        )
    print(f"✅ Hoàn tất tính toán trước trong {time.time() - t0:.1f} giây!")
    
    # 3. GRID SEARCH QUÉT MA TRẬN TRỌNG SỐ TRONG RAM (CHỈ MẤT ~0.05 - 0.5 GIÂY CHO 70 TỔ HỢP)
    RERANK_TOP_K_VALUES = [10, 15, 20, 25, 30, 40, 50]
    ST1_WEIGHT_VALUES = [0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00]
    
    print(f"\n⚡ [Bước 2/2] Quét ma trận {len(RERANK_TOP_K_VALUES) * len(ST1_WEIGHT_VALUES)} tổ hợp tham số trong RAM...")
    results_grid = []
    best_recall = 0.0
    best_config = {}
    
    t_grid = time.time()
    for rtk in RERANK_TOP_K_VALUES:
        for w in ST1_WEIGHT_VALUES:
            preds = {}
            for qid in val_items_eval:
                preds[qid] = hybrid_gs.fuse_from_cached_scores(
                    cached_val_items[qid],
                    top_k=5,
                    rerank_top_k=rtk,
                    st1_weight=w
                )
    
            metrics = compute_metrics(preds, val_truth_eval, k=5)
            recall = metrics["Recall"]
            precision = metrics["Precision"]
    
            results_grid.append({
                "rerank_top_k": rtk,
                "st1_weight": w,
                "recall": recall,
                "precision": precision
            })
    
            if recall > best_recall:
                best_recall = recall
                best_config = {"rerank_top_k": rtk, "st1_weight": w, "recall": recall, "precision": precision}
    
    print(f"🎉 Đã hoàn tất duyệt {len(results_grid)} tổ hợp chỉ trong {time.time() - t_grid:.2f} giây!")
    
    # === BẢNG TỔNG KẾT GRID SEARCH ===
    print("\n" + "=" * 80)
    print("📊 BẢNG TỔNG KẾT GRID SEARCH (RECALL@5 %)")
    print("=" * 80)
    
    header = f"{'rerank_top_k':>13}"
    for w in ST1_WEIGHT_VALUES:
        header += f" | w={w:.2f}"
    print(header)
    print("-" * len(header))
    
    for rtk in RERANK_TOP_K_VALUES:
        row = f"{rtk:>13}"
        for w in ST1_WEIGHT_VALUES:
            r = [x for x in results_grid if x["rerank_top_k"] == rtk and x["st1_weight"] == w][0]
            row += f" | {r['recall']*100:5.2f}%"
        print(row)
    
    # Top 5 cấu hình xuất sắc nhất
    sorted_results = sorted(results_grid, key=lambda x: (x["recall"], x["precision"]), reverse=True)
    print("\n🏆 TOP 5 TỔ HỢP ĐẠT ĐIỂM CAO NHẤT:")
    print("-" * 75)
    for i, r in enumerate(sorted_results[:5]):
        print(f"  #{i+1}: rerank_top_k={r['rerank_top_k']:3d} | st1_weight={r['st1_weight']:.2f} | "
              f"Recall={r['recall']*100:.2f}% | Precision={r['precision']*100:.2f}%")
    
    print(f"\n⭐ CẤU HÌNH TỐI ƯU CHIẾN THẮNG:")
    print(f"   👉 BEST_RERANK_TOP_K = {best_config['rerank_top_k']}")
    print(f"   👉 BEST_ST1_WEIGHT   = {best_config['st1_weight']:.2f}")
    print(f"   👉 KẾT QUẢ ĐẠT ĐƯỢC   : Recall@5 = {best_config['recall']*100:.2f}% | Precision@5 = {best_config['precision']*100:.2f}%")
    print("=" * 80)
    
    BEST_RERANK_TOP_K = best_config["rerank_top_k"]
    BEST_ST1_WEIGHT = best_config["st1_weight"]
    print(f"💾 Đã lưu cấu hình tối ưu để Cell 17 tự động sinh file nộp bài submission_finetuned.zip!")
    





## 📦 CELL 16: Hướng dẫn Sinh Submission Tối Ưu từ Kết Quả Grid Search



In [ ]:
# ==============================================================================
# 📌 CELL 17: Sinh File Nộp Bài submission_finetuned.zip (Cấu hình Tối ưu Cell 15)
# ==============================================================================
if BUILD_PKL_CACHE_ONLY:
    print("⏭️ Đã bỏ qua sinh submission vì BUILD_PKL_CACHE_ONLY = True.")
else:
    # === Sinh submission_finetuned.zip với cấu hình tối ưu từ Grid Search ===
    import json
    import zipfile
    from tqdm import tqdm
    
    print(f"🚀 Sinh submission với cấu hình tối ưu:")
    print(f"   rerank_top_k = {BEST_RERANK_TOP_K}")
    print(f"   st1_weight   = {BEST_ST1_WEIGHT}")
    
    # Dự đoán trên toàn bộ Public Test
    test_preds_best = {}
    for qid, item in tqdm(test_data.items(), desc="Predicting (Best Config)"):
        question = item["question"]
        test_preds_best[qid] = hybrid_gs.search(
            question, top_k=5,
            rerank_top_k=BEST_RERANK_TOP_K,
            st1_weight=BEST_ST1_WEIGHT
        )
    
    # Định dạng và đóng gói
    formatted = {}
    for qid, docs in test_preds_best.items():
        formatted[str(qid)] = {"answer": [str(d) for d in docs[:5]]}
    
    with open("submission_finetuned.json", "w", encoding="utf-8") as f:
        json.dump(formatted, f, ensure_ascii=False, indent=4)
    
    with zipfile.ZipFile("submission_finetuned.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write("submission_finetuned.json", arcname="submission.json")
    
    size_kb = os.path.getsize("submission_finetuned.zip") / 1024
    print(f"\n🎉 ĐÃ TẠO: 'submission_finetuned.zip' ({size_kb:.1f} KB)")
    print(f"   Config: rerank_top_k={BEST_RERANK_TOP_K}, st1_weight={BEST_ST1_WEIGHT}")

